# 4a. Multiple Linear Regression 
**Cut-off Date: 2026-05-26**

This notebook formalises the predictive modeling phase of the project, establishing a baseline performance benchmark through Multiple Linear Regression (MLR). By systematically applying chronological train-test partitioning and sequential cross-validation, this document captures the core structural dependencies in airline pricing, providing a quantitative foundation for evaluating feature importance.

## Objectives
- Chronological Validation Strategy: Implement a temporal train-test split to ensure model evaluation respects data sequence, effectively eliminating look-ahead bias and simulating real-world predictive scenarios.
- Baseline Performance Benchmarking: Execute a Multiple Linear Regression (MLR) baseline to quantify initial predictive accuracy and establish a performance benchmark for subsequent complex ensemble modeling.
- Iterative Robustness Testing: Apply TimeSeriesSplit cross-validation to rigorously assess model stability across successive historical data folds, ensuring performance consistency despite variance in flight pricing.
- Feature Interpretability & Driver Analysis: Extract and rank model coefficients to identify primary market drivers.

## Model Performance and Success Benchmarks

1. **Mean Absolute Percentage Error (MAPE) < 25.0%**

   Measures error as a percentage rather than a raw dollar amount, allowing us to maintain a consistent standard of accuracy across both low-cost, short-haul flights and premium, long-haul sectors.
      
2. **Coefficient of Determination (R^2) > 85.0%**

    Demonstrates that over 85% of price variance is successfully captured by our feature engineering. This confirms the model is learning the true market drivers rather than just fitting to noise.
   
3. **Root Mean Square Error (RMSE) < $300**

   Sets a firm boundary on our absolute error, ensuring that even our "worst-case" predictions remain actionable for a consumer planning their travel budget.

### Environment Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

In [2]:
df = pd.read_csv('./dataset - base (2026-05-26).csv')

### Data Processing

In [3]:
# Handle date formatting and sorting them chronologically
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values('date').reset_index(drop = True)

In [4]:
# Boolean sanitisation - ensure that all boolean fields are shown as 0 or 1. 
for col in ['is_weekend', 'is_lcc', 'is_holiday_sin', 'is_holiday_other', 'is_sch_holiday']:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.upper().map({'TRUE': 1, 'FALSE': 0, '1': 1, '0': 0})
    else:
        df[col] = df[col].astype(int)

In [5]:
# Extract departure month from deaprture date for seasonal features 
df['departure_date'] = pd.to_datetime(df['departure_date'], errors='coerce')
df['departure_month'] = df['departure_date'].dt.month

In [6]:
# Check data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1240 entries, 0 to 1239
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               193 non-null    datetime64[ns]
 1   route              1240 non-null   object        
 2   departure_date     664 non-null    datetime64[ns]
 3   price              1240 non-null   float64       
 4   days_to_departure  1240 non-null   int64         
 5   day_of_week        1240 non-null   int64         
 6   day_name           1240 non-null   object        
 7   is_weekend         1240 non-null   int64         
 8   departure_airport  1240 non-null   object        
 9   out_inbound        1240 non-null   int64         
 10  other_airport      1240 non-null   object        
 11  data_source        1240 non-null   object        
 12  booking_window     1240 non-null   object        
 13  airline            1240 non-null   object        
 14  airline_

### Feature Selection and Encoding

In [7]:
# Selecting features and one-hot encoding
core_features = [
    'booking_window',
    'day_of_week',
    'is_weekend',
    'is_lcc',
    'is_holiday_sin',
    'is_holiday_other',
    'is_sch_holiday',
    'route',
    'departure_month']

X = pd.get_dummies(df[core_features], columns = ['route','booking_window','departure_month'], drop_first = True)
y = df['price']

### Model Training and Coefficient Analysis

#### Chronological Train/Test Partitioning

By splitting chronologically rather than randomly, we simulate a real-world predictive scenario where the model is trained on past pricing patterns to forecast future fares, thereby eliminating look-ahead leakage.

In [8]:
# 80% train data, 20% test data.

split_idx = int(len(df) * 0.8)
X_dev, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_dev, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Total historical matrix depth: {X.shape[0]} records")
print(f"Development pool (80%): {X_dev.shape[0]} records")
print(f"Hidden validation pool (20%): {X_test.shape[0]} records") 

Total historical matrix depth: 1240 records
Development pool (80%): 992 records
Hidden validation pool (20%): 248 records


#### Sequential Cross Validation: Evaluating Model Robustness

By applying TimeSeriesSplit sequentially, we ensure each validation fold is performed on a "future" window relative to the training window, providing an unbiased estimate of our model's performance on unseen temporal data.

In [9]:
# In the 80% train data, it generates sequential index pairs (train_idx and val_idx) across 5 successive folds. 
tscv = TimeSeriesSplit(n_splits =5)
cv_mae_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_dev)):
    X_cv_train, X_cv_val = X_dev.iloc[train_idx], X_dev.iloc[val_idx]
    y_cv_train, y_cv_val = y_dev.iloc[train_idx], y_dev.iloc[val_idx]

    scaler = StandardScaler()
    X_cv_train_scaled = scaler.fit_transform(X_cv_train)
    X_cv_val_scaled = scaler.transform(X_cv_val)
    
    fold_mlr = LinearRegression()
    fold_mlr.fit(X_cv_train_scaled, y_cv_train)
    fold_pred = fold_mlr.predict(X_cv_val_scaled)
    cv_mae_scores.append(mean_absolute_error(y_cv_val, fold_pred))

    print(f"  Fold {fold+1} Validation MAE: {cv_mae_scores[-1]:.2f} SGD")

print(f"\nMean Inner CV MAE: {np.mean(cv_mae_scores):.2f} SGD\n")

  Fold 1 Validation MAE: 135.98 SGD
  Fold 2 Validation MAE: 95.53 SGD
  Fold 3 Validation MAE: 477.93 SGD
  Fold 4 Validation MAE: 515.37 SGD
  Fold 5 Validation MAE: 335.50 SGD

Mean Inner CV MAE: 312.06 SGD



#### Final Model Evaluation: Performance Assessment on Holdout Data

In [10]:
scaler = StandardScaler()
X_dev_scaled = scaler.fit_transform(X_dev)
X_test_scaled = scaler.transform(X_test)

mlr_master = LinearRegression()
mlr_master.fit(X_dev_scaled, y_dev)
holdout_predictions = mlr_master.predict(X_test_scaled)

Holdout Mean Absolute Error (MAE):   394.70 SGD
Holdout Root Mean Squared Error (RMSE): 513.08 SGD
Holdout Variance Coverage (R² Score):   0.5547 (55.47%)
Holdout Mean Absolute Percentage Error (MAPE): 63.29%


##### Metrics

In [ ]:
holdout_mae = mean_absolute_error(y_test, holdout_predictions)
holdout_rmse = np.sqrt(mean_squared_error(y_test, holdout_predictions))
holdout_r2 = r2_score(y_test, holdout_predictions)
holdout_mape = mean_absolute_percentage_error(y_test, holdout_predictions)

print(f"Holdout Mean Absolute Error (MAE):   {holdout_mae:.2f} SGD")
print(f"Holdout Root Mean Squared Error (RMSE): {holdout_rmse:.2f} SGD")
print(f"Holdout Variance Coverage (R² Score):   {holdout_r2:.4f} ({holdout_r2*100:.2f}%)")
print(f"Holdout Mean Absolute Percentage Error (MAPE): {holdout_mape*100:.2f}%") 

#### Analysis

The evaluation of our Multiple Linear Regression (MLR) baseline confirms that airfare pricing dynamics are non-linear and significantly more complex than a standard regression framework can capture.

##### Model Underfitting
The holdout $R^2$ of 55.47% indicates that the baseline MLR model fails to explain approximately 45% of price variance. This suggests that the relationship between our features and final fares involves non-linear "staircase" pricing effects which the MLR's linear assumption cannot internalize.

##### Generalisation Gap
We observed a performance degradation between our sequential cross-validation (MAE: 312.88 SGD) and the production holdout test (MAE: 394.70 SGD). This 26% increase in error demonstrates that the model suffers from high bias, failing to adapt to the pricing volatility inherent in unseen, future temporal data.

##### Error Magnitude
The high MAPE of 63.29% provides further quantitative evidence that the linear approach lacks the necessary sensitivity to map the complex, demand-driven pricing strategies employed by modern carriers.

#### Coefficient Analysis: Quantifying Feature Impact on Fare Pricing

In [12]:
# Coefficient Extraction 
print(f"Intercept: {mlr_master.intercept_:.2f}")

coefficients = pd.DataFrame({
    'Feature_Attribute': X_dev.columns,
    'Coefficient': mlr_master.coef_
})

coefficients['Abs_Coefficient'] = coefficients['Coefficient'].abs()

coefficients = coefficients.sort_values(by='Abs_Coefficient', ascending=False)

#Top 5 features
coefficients.head(5)

Intercept: 742.40


,Feature_Attribute,Coefficient,Abs_Coefficient
10,route_SIN-LHR,392.161444,392.161444
6,route_LHR-SIN,287.287098,287.287098
2,is_lcc,-188.121555,188.121555
8,route_NRT-SIN,170.275118,170.275118
12,route_SIN-NRT,158.884843,158.884843
